# 📊 Model Comparison & Explainability
## Disease Prediction System

This notebook:
1. Creates comprehensive model ranking tables
2. Generates publication-quality comparison charts
3. Performs SHAP explainability analysis

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.config import MODEL_RESULTS_DIR, FIGURES_DIR
from src.data_loader import load_dataset
from src.preprocessing import PreprocessingPipeline
from src.train import train_all_models
from src.evaluate import evaluate_all_models, create_comparison_table
from src.explainability import (
    compute_feature_importance, compute_shap_values,
    plot_feature_importance, plot_shap_summary,
    generate_explainability_report
)

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup complete!')

## 1. Load All Comparison Results

In [ ]:
datasets_info = {
    'heart': 'Heart Disease',
    'diabetes': 'Diabetes',
    'cancer': 'Breast Cancer'
}

all_comparisons = {}

for key, name in datasets_info.items():
    path = MODEL_RESULTS_DIR / f'{key}_comparison.csv'
    if path.exists():
        df = pd.read_csv(path)
        df['Dataset'] = name
        all_comparisons[key] = df
        print(f"\n--- {name} ---")
        display(df)
    else:
        print(f"Results not found for {name}. Run training notebook first.")

## 2. Combined Ranking Table

In [ ]:
if all_comparisons:
    combined_df = pd.concat(all_comparisons.values(), ignore_index=True)
    combined_df = combined_df[['Dataset', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']]
    combined_df = combined_df.sort_values(['Dataset', 'ROC-AUC'], ascending=[True, False])
    
    print("\n" + "="*80)
    print("COMPLETE MODEL RANKING TABLE")
    print("="*80)
    display(combined_df)
    
    # Save combined results
    combined_df.to_csv(MODEL_RESULTS_DIR / 'combined_ranking.csv', index=False)
    print("\nSaved: combined_ranking.csv")

## 3. Best Model per Dataset

In [ ]:
if all_comparisons:
    print("\n🏆 BEST MODELS (by ROC-AUC):")
    print("-" * 50)
    
    best_models = []
    for key, df in all_comparisons.items():
        best_idx = df['ROC-AUC'].idxmax()
        best_row = df.loc[best_idx]
        best_models.append({
            'Dataset': datasets_info[key],
            'Best Model': best_row['Model'],
            'Accuracy': best_row['Accuracy'],
            'ROC-AUC': best_row['ROC-AUC'],
            'F1 Score': best_row['F1 Score']
        })
        print(f"  {datasets_info[key]}: {best_row['Model']} (ROC-AUC: {best_row['ROC-AUC']:.4f})")
    
    best_df = pd.DataFrame(best_models)
    display(best_df)

## 4. Publication-Quality Comparison Charts

In [ ]:
if all_comparisons:
    # Grouped bar chart by dataset
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for idx, (key, df) in enumerate(all_comparisons.items()):
        metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
        x = np.arange(len(df))
        width = 0.15
        
        for i, metric in enumerate(metrics):
            axes[idx].bar(x + i * width, df[metric], width, label=metric)
        
        axes[idx].set_xlabel('Model')
        axes[idx].set_ylabel('Score')
        axes[idx].set_title(datasets_info[key], fontsize=13, fontweight='bold')
        axes[idx].set_xticks(x + width * 2)
        axes[idx].set_xticklabels(df['Model'], rotation=20, ha='right', fontsize=9)
        axes[idx].set_ylim(0, 1.1)
        axes[idx].legend(fontsize=8, loc='lower right')
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Model Performance Comparison Across Datasets', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'combined_model_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: combined_model_comparison.png')

In [ ]:
if all_comparisons:
    # ROC-AUC comparison across datasets
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = ['Logistic Regression', 'SVM', 'Random Forest', 'XGBoost']
    x = np.arange(len(models))
    width = 0.25
    
    for i, (key, df) in enumerate(all_comparisons.items()):
        auc_values = []
        for model in models:
            row = df[df['Model'] == model]
            if not row.empty:
                auc_values.append(row['ROC-AUC'].values[0])
            else:
                auc_values.append(0)
        ax.bar(x + i * width, auc_values, width, label=datasets_info[key])
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('ROC-AUC Score', fontsize=12)
    ax.set_title('ROC-AUC Comparison Across Datasets', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(models)
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'roc_auc_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: roc_auc_comparison.png')

## 5. SHAP Explainability Analysis

In [ ]:
# Run explainability for each dataset using the best model
for dataset_name in ['heart', 'diabetes', 'cancer']:
    print(f"\n{'='*60}")
    print(f"EXPLAINABILITY: {datasets_info[dataset_name]}")
    print(f"{'='*60}")
    
    try:
        # Load and preprocess
        df = load_dataset(dataset_name)
        pipeline = PreprocessingPipeline()
        X_train, X_test, y_train, y_test = pipeline.fit_transform(df)
        feature_names = pipeline.feature_names
        
        # Train models
        models = train_all_models(
            X_train, y_train, dataset_name,
            tune_hyperparameters=False, save_models=False
        )
        
        # Generate explainability report
        report = generate_explainability_report(
            models, X_train, X_test, feature_names, dataset_name
        )
        
        print(f"Explainability report generated for {dataset_name}")
        
    except Exception as e:
        print(f"Error: {e}")

## 6. Summary & Conclusions

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON & EXPLAINABILITY COMPLETE")
print("="*60)
print("\nAll comparison charts and SHAP plots saved to reports/figures/")
print("Ranking tables saved to reports/model_results/")
print("\nKey outputs:")
print("  - combined_model_comparison.png")
print("  - roc_auc_comparison.png")
print("  - SHAP summary plots per dataset/model")
print("  - Feature importance plots")
print("  - combined_ranking.csv")